In [ ]:
#For practical reasons im gonna import all of the libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

ModuleNotFoundError: No module named 'Ipython'

In [12]:
#We read the dataset

drug_use_dataset= pd.read_csv('../data/drug_consumption.data')

#We set column names cause the data itself doesnt have headers

drug_use_dataset.columns= [
    'ID',
    'Age',
    'Gender',
    'Education',
    'Country',
    'Ethnicity',
    'Nscore',
    'Escore',
    'Oscore',
    'Ascore',
    'Cscore',
    'Impulsive',
    'Ss',
    'Alcohol',
    'Amphet',
    'Amyl',
    'Benzos',
    'Caff',
    'Cannabis',
    'Choc',
    'Coke',
    'Crack',
    'Ecstacy',
    'Heroin',
    'Ketamine',
    'Legalh',
    'Lsd',
    'Meth',
    'Mushrooms',
    'Nicotine',
    'Semer',
    'Vsa',
]

#First I'm gonna drop the ID value which doesnt have any properties to study:

drug_use_dataset = drug_use_dataset.drop('ID', axis=1)

#And now to show me the first rows to know is it working properly

drug_use_dataset.head()

#To clarify:

#CL0 = Never used
#CL1 = Used over a decade ago
#CL2 = Used in last decade
#CL3 = Used in last year
#CL4 = Used in last month
#CL5 = Used in last week
#CL6 = Used yesterday

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,...,Ecstacy,Heroin,Ketamine,Legalh,Lsd,Meth,Mushrooms,Nicotine,Semer,Vsa
0,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,-0.14277,...,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
1,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,-1.01450,...,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
2,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,0.58489,...,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
3,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,1.30612,...,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0
4,2.59171,0.48246,-1.22751,0.24923,-0.31685,-0.67825,-0.30033,-1.55521,2.03972,1.63088,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL6,CL0,CL0


In [ ]:
#I'm gonna split the data from the targets

inputs = drug_use_dataset.drop(['Alcohol',
    'Amphet',
    'Amyl',
    'Benzos',
    'Caff',
    'Cannabis',
    'Choc',
    'Coke',
    'Crack',
    'Ecstacy',
    'Heroin',
    'Ketamine',
    'Legalh',
    'Lsd',
    'Meth',
    'Mushrooms',
    'Nicotine',
    'Semer',
    'Vsa',], axis=1)

outputs = drug_use_dataset[['Alcohol',
    'Amphet',
    'Amyl',
    'Benzos',
    'Caff',
    'Cannabis',
    'Choc',
    'Coke',
    'Crack',
    'Ecstacy',
    'Heroin',
    'Ketamine',
    'Legalh',
    'Lsd',
    'Meth',
    'Mushrooms',
    'Nicotine',
    'Semer',
    'Vsa']]

#Lets see them:

inputs.head()


,Alcohol,Amphet,Amyl,Benzos,Caff,Cannabis,Choc,Coke,Crack,Ecstacy,Heroin,Ketamine,Legalh,Lsd,Meth,Mushrooms,Nicotine,Semer,Vsa
0,CL5,CL2,CL2,CL0,CL6,CL4,CL6,CL3,CL0,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
1,CL6,CL0,CL0,CL0,CL6,CL3,CL4,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
2,CL4,CL0,CL0,CL3,CL5,CL2,CL4,CL2,CL0,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
3,CL4,CL1,CL1,CL0,CL6,CL3,CL6,CL0,CL0,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0
4,CL2,CL0,CL0,CL0,CL6,CL0,CL4,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL6,CL0,CL0


In [19]:
outputs.head()

,Alcohol,Amphet,Amyl,Benzos,Caff,Cannabis,Choc,Coke,Crack,Ecstacy,Heroin,Ketamine,Legalh,Lsd,Meth,Mushrooms,Nicotine,Semer,Vsa
0,CL5,CL2,CL2,CL0,CL6,CL4,CL6,CL3,CL0,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
1,CL6,CL0,CL0,CL0,CL6,CL3,CL4,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
2,CL4,CL0,CL0,CL3,CL5,CL2,CL4,CL2,CL0,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
3,CL4,CL1,CL1,CL0,CL6,CL3,CL6,CL0,CL0,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0
4,CL2,CL0,CL0,CL0,CL6,CL0,CL4,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL6,CL0,CL0


In [ ]:
#Okay with all of the above already done we can start studying the data

#Working with a dataset that is classificated we have to use the stratified method:

#First I'm gonna calculate the mean

inputs_norm = (inputs - inputs.mean())/inputs.std

